# Beam search — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
DIRS=[(1,0),(-1,0),(0,1),(0,-1)]
def neighbors(p, shape=(5,5), walls=set()):
    for dr,dc in DIRS:
        q=(p[0]+dr,p[1]+dc)
        if 0<=q[0]<shape[0] and 0<=q[1]<shape[1] and q not in walls: yield q
def reconstruct(parent, goal):
    path=[]; x=goal
    while x is not None: path.append(x); x=parent.get(x)
    return path[::-1]
def draw_grid(path=(), explored=(), frontier=(), walls=set(), start=(0,0), goal=(4,4), shape=(5,5), title='grid'):
    img=np.zeros(shape)
    for r,c in walls: img[r,c]=-1
    for r,c in explored: img[r,c]=.35
    for r,c in frontier: img[r,c]=.65
    for r,c in path: img[r,c]=1
    plt.figure(figsize=(4,4)); plt.imshow(img,cmap='viridis',vmin=-1,vmax=1); plt.scatter([start[1]],[start[0]],c='w',edgecolor='k'); plt.scatter([goal[1]],[goal[0]],c='r',edgecolor='k'); plt.title(title); plt.xticks(range(shape[1])); plt.yticks(range(shape[0])); plt.grid(color='w',alpha=.3)
print('setup ok')

## A fixed-width frontier cap
Beam search is best-first search with deliberate amnesia: only k candidates survive each layer.

In [ ]:
levels=[[('',0)],[('A',5),('B',4)],[('AA',5),('AB',6),('BA',10),('BB',1)]]; beam=[levels[0][0]]
for d in [1,2]: beam=sorted([x for x in levels[d] if x[0].startswith(beam[0][0])],key=lambda z:-z[1])[:1]
plt.figure(figsize=(5,3)); plt.bar([beam[0][0]],[beam[0][1]]); plt.title('width 1 greedy')
assert beam[0]==('AB',6)

In [ ]:
levels=[[('',0)],[('A',5),('B',4)],[('AA',5),('AB',6),('BA',10),('BB',1)]]; beam=[('',0)]
for d in [1,2]:
 cand=[]
 for pref,_ in beam: cand += [x for x in levels[d] if x[0].startswith(pref)]
 beam=sorted(cand,key=lambda z:-z[1])[:2]
plt.figure(figsize=(5,3)); plt.bar([x[0] for x in beam],[x[1] for x in beam]); plt.title('width 2')
assert beam[0]==('BA',10)

In [ ]:
ks=[1,2,3,4]; plt.figure(figsize=(5,3)); plt.plot(ks,[min(k,4) for k in ks],'-o'); plt.title('memory cap')
assert [min(k,4) for k in ks]==[1,2,3,4]

In [ ]:
seq=[('short',6,2),('long',9,5),('best',8,3)]; raw=[s for _,s,_ in seq]; norm=[s/l for _,s,l in seq]
plt.figure(figsize=(5,3)); x=np.arange(3); plt.bar(x-.15,raw,.3); plt.bar(x+.15,norm,.3); plt.xticks(x,[n for n,_,_ in seq]); plt.title('normalization')
assert seq[int(np.argmax(raw))][0]=='long' and seq[int(np.argmax(norm))][0]=='short'

In [ ]:
s=(0,0); g=(4,4); walls={(0,1),(1,1),(2,1),(3,1)}
def bg(k):
 beam=[(s,[s])]
 for _ in range(12):
  cand=[]
  for u,p in beam:
   if u==g: return p
   for v in neighbors(u,walls=walls):
    if v not in p: cand.append((v,p+[v]))
  beam=sorted(cand,key=lambda z:abs(z[0][0]-g[0])+abs(z[0][1]-g[1]))[:k]
 return []
p1=bg(1); p2=bg(2); draw_grid(p2,p2,walls=walls,title='beam on grid')
assert p1[-1]==g and p2[-1]==g